# Palmer Penguins — Statistical Models

> **Final Project · UFS-06/07**

The Palmer Penguins dataset contains morphological measurements of **333 penguins** from three species in the Palmer Archipelago, Antarctica.

| Variable | Type | Description |
|---|---|---|
| `bill_length_mm` | continuous | Bill length (mm) |
| `bill_depth_mm` | continuous | Bill depth (mm) |
| `flipper_length_mm` | continuous | Flipper length (mm) |
| `body_mass_g` | continuous | Body mass (g) |
| `species` | categorical | Adélie · Chinstrap · Gentoo |
| `island` | categorical | Biscoe · Dream · Torgersen |

## Project Structure

| # | Question | Tool |
|---|---|---|
| 1 | Who are the Palmer Penguins? | EDA, descriptive statistics |
| 2 | Can we assume normality in the data? | Shapiro-Wilk, Q-Q plot |
| 3 | How heavy are the penguins, really? | 95% CI, t-Student |
| 4 | Does each species have its own island? | Chi-square |
| 5 | Does the island affect the weight of Adélie penguins? | ANOVA, Levene, Tukey HSD |
| 6 | Does the bill reveal the species? Simpson's Paradox | Pearson Correlation |
| 7 | Can we tell which species a penguin belongs to just by looking at it? | Logistic Regression |

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from scipy.stats import shapiro, levene, f_oneway, chi2_contingency
from scipy.stats import gaussian_kde
from matplotlib.lines import Line2D
from IPython.display import display
import matplotlib.colors as mcolors
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings("ignore")

# ── Unified color palette for all plots ──────────────────────────────────────
# Three neutral colors for species — distinguishable in color and grayscale
COLORS = {
    "Adelie"   : "#4C72B0",   # slate blue
    "Chinstrap": "#C44E52",   # terracotta
    "Gentoo"   : "#55A868",   # sage green
}
SPECIES_ORDER = ["Adelie", "Chinstrap", "Gentoo"]

# Auxiliary colors — used in CI bars, reference lines, etc.
C_DARK  = "#2C2C2A"   # near black — CI lines, axes
C_MID   = "#888780"   # medium gray — secondary points, grid
C_LIGHT = "#F1EFE8"   # light gray — figure backgrounds

# Global significance level
ALPHA = 0.05

# Continuous morphological variables
MORPH = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

# Global plot style
plt.rcParams.update({
    "figure.facecolor" : C_LIGHT,
    "axes.facecolor"   : "white",
    "axes.edgecolor"   : C_MID,
    "axes.labelcolor"  : C_DARK,
    "axes.titlecolor"  : C_DARK,
    "axes.titlesize"   : 11,
    "axes.labelsize"   : 10,
    "axes.grid"        : True,
    "grid.color"       : "#D3D1C7",
    "grid.alpha"       : 0.5,
    "grid.linewidth"   : 0.6,
    "xtick.color"      : C_MID,
    "ytick.color"      : C_MID,
    "text.color"       : C_DARK,
    "legend.framealpha": 0.92,
    "legend.edgecolor" : "#D3D1C7",
    "figure.dpi"       : 110,
    "font.family"      : "sans-serif",
})

print("Setup complete.")

## 1. Who Are the Palmer Penguins?
### Exploratory Data Analysis (EDA)

Before applying any statistical test, we need to understand the data: how many observations, what they measure, whether there are missing values, and what differences are visible at a glance between the three species.

**Visual exploration is the first step of any serious analysis.** The patterns we identify here will guide all methodological decisions in the following sections.

In [ ]:
# Load dataset and drop the 9 rows with unrecorded sex.
# Sex is a key stratification variable in later tests,
# so incomplete rows would generate empty subgroups.
df = pd.read_csv("penguins.csv")
df = df.dropna(subset=["sex"]).reset_index(drop=True)

print(f"Rows × columns: {df.shape[0]} × {df.shape[1]}")
print(f"Species: {sorted(df['species'].unique())}")
print(f"Islands : {sorted(df['island'].unique())}")
print()
df.head(8)

In [ ]:
# Dataset structure: column types and non-null counts
df.info()

In [ ]:
# Descriptive statistics: mean, standard deviation, quartiles.
# Note the range of body_mass_g — group differences are already apparent.
df.describe().round(2)

In [ ]:
# Distribution by species and sex
print("Count by species:")
print(df["species"].value_counts().to_string(), "\n")

print("Count by sex:")
print(df["sex"].value_counts().to_string(), "\n")

print("Species × sex table — group balance:")
print(pd.crosstab(df["species"], df["sex"]))

In [ ]:
# Morphological averages by species and sex.
# This table is the first quantitative indication of inter-group differences.
df.groupby(["species", "sex"])[MORPH].mean().round(1)

In [ ]:
# Exploratory visualization — four complementary perspectives
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.flatten()

# ── 1. Mean body mass by species — horizontal bars ───────────────────────────
medias = df.groupby("species")["body_mass_g"].mean().reindex(SPECIES_ORDER)
bars = axes[0].barh(
    SPECIES_ORDER, medias.values,
    color=[COLORS[s] for s in SPECIES_ORDER],
    height=0.5, zorder=2
)
for bar, val in zip(bars, medias.values):
    axes[0].text(val + 40, bar.get_y() + bar.get_height() / 2,
                 f"{int(val):,} g", va="center", ha="left",
                 fontsize=10, color="black", fontweight="bold")
axes[0].set_xlim(0, 6200)
axes[0].set_xlabel("Body mass (g)")
axes[0].set_title("Mean body mass by species")
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x):,}"))
axes[0].spines[["top", "right"]].set_visible(False)

# ── 2. Flipper length distribution — KDE curves ───────────────────────────────
for sp in SPECIES_ORDER:
    data = df[df["species"] == sp]["flipper_length_mm"].dropna()
    data.plot.kde(ax=axes[1], color=COLORS[sp], label=sp,
                   linewidth=2.2, bw_method=0.35)
    xs = np.linspace(data.min(), data.max(), 300)
    axes[1].fill_between(xs, gaussian_kde(data, bw_method=0.35)(xs),
                         alpha=0.12, color=COLORS[sp])
axes[1].set_xlabel("Flipper length (mm)")
axes[1].set_ylabel("Density")
axes[1].set_title("Distribution — flipper length")
axes[1].set_yticks([])
axes[1].legend(frameon=False, fontsize=9)
axes[1].spines[["top", "right"]].set_visible(False)

# ── 3. Bill length distribution — KDE curves ─────────────────────────────────
for sp in SPECIES_ORDER:
    data = df[df["species"] == sp]["bill_length_mm"].dropna()
    data.plot.kde(ax=axes[2], color=COLORS[sp], label=sp,
                   linewidth=2.2, bw_method=0.35)
    xs = np.linspace(data.min(), data.max(), 300)
    axes[2].fill_between(xs, gaussian_kde(data, bw_method=0.35)(xs),
                         alpha=0.12, color=COLORS[sp])
axes[2].set_xlabel("Bill length (mm)")
axes[2].set_ylabel("Density")
axes[2].set_title("Distribution — bill length")
axes[2].set_yticks([])
axes[2].legend(frameon=False, fontsize=9)
axes[2].spines[["top", "right"]].set_visible(False)

# ── 4. Flipper vs. mass with trend line per species ───────────────────────────
for sp in SPECIES_ORDER:
    sub = df[df["species"] == sp]
    axes[3].scatter(sub["flipper_length_mm"], sub["body_mass_g"],
                    color=COLORS[sp], alpha=0.50, s=26, label=sp,
                    edgecolors="white", linewidths=0.4)
    m, b = np.polyfit(sub["flipper_length_mm"], sub["body_mass_g"], 1)
    xs = np.linspace(sub["flipper_length_mm"].min(), sub["flipper_length_mm"].max(), 100)
    axes[3].plot(xs, m * xs + b, color=COLORS[sp], linewidth=1.8, alpha=0.9)
axes[3].set_xlabel("Flipper length (mm)")
axes[3].set_ylabel("Body mass (g)")
axes[3].set_title("Flipper vs. mass — trend by species")
axes[3].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x):,}"))
axes[3].legend(frameon=False, fontsize=9)
axes[3].spines[["top", "right"]].set_visible(False)

fig.suptitle("Exploratory Analysis — Palmer Penguins",
             fontsize=13, fontweight="bold")
plt.tight_layout(pad=2.0)
plt.show()

### Conclusion — EDA

- **Body mass:** Gentoo penguins (~5,092 g) are significantly heavier than Adélie (~3,706 g) and Chinstrap (~3,733 g), which are nearly identical to each other.

- **Flipper length:** The Gentoo distribution is completely separated to the right (~217 mm vs ~190 mm for the other two species). This separation is so clean that flipper length is the strongest morphological signal for species identification.

- **Bill length:** Adélie penguins have a noticeably shorter bill (~38 mm), while Chinstrap and Gentoo overlap around ~47 mm.

- **Flipper–mass correlation:** Within each species there is a clear positive relationship — penguins with longer flippers tend to be heavier. This pattern holds consistently within each species.

## 2. Can We Assume Normality in the Data?
### Shapiro-Wilk Test and Q-Q Plots

The tests used in the following sections — t-test, ANOVA, regression — are **parametric tests**: they assume that the data within each group follow a normal distribution. If that assumption is severely violated, the conclusions may be incorrect.

We therefore verify normality before proceeding, using two methods:

| Method | Type | How to read it |
|---|---|---|
| Shapiro-Wilk | Analytical | Returns a p-value. If p > 0.05 — we do not reject normality |
| Q-Q plot | Graphical | If points follow the diagonal line — normal distribution |

The correct stratification is **species × sex**: subsequent tests compare subgroups, so normality must hold within them.

**Hypotheses:**
- H₀: the data follow a normal distribution
- H₁: the data do not follow a normal distribution
- Significance level: α = 0.05

In [ ]:
# Shapiro-Wilk test — stratified by species × sex
print(f"Shapiro-Wilk Test  |  H₀: normal distribution  |  α = {ALPHA}\n")
print(f"{'Species':<12} {'Sex':<8} {'Variable':<22} {'W':>7}  {'p-value':>9}  Result")
print("─" * 72)

non_normal_cases = []

for sp in sorted(df["species"].unique()):
    for sx in ["MALE", "FEMALE"]:
        subset = df[(df["species"] == sp) & (df["sex"] == sx)]
        for var in MORPH:
            W, p = shapiro(subset[var].dropna())
            result = "NOT normal" if p < ALPHA else "Normal"
            print(f"{sp:<12} {sx:<8} {var:<22} {W:>7.4f}  {p:>9.4f}  {result}")
            if p < ALPHA:
                non_normal_cases.append((sp, sx, var, p))
    print()

print(f"Cases rejecting H₀: {len(non_normal_cases)}")

In [ ]:
# Q-Q plots only for non-normal cases.
# The Q-Q plot compares the empirical quantiles of our data
# with those of a theoretical normal. If points align with the diagonal — normal.
# Deviations in the tails indicate skewness or heavy tails.

cases = [
    ("Adelie",    "MALE",   "bill_depth_mm"),
    ("Chinstrap", "FEMALE", "bill_length_mm"),
    ("Gentoo",    "MALE",   "bill_length_mm"),
]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, (sp, sx, var) in zip(axes, cases):
    sub   = df[(df["species"] == sp) & (df["sex"] == sx)][var].dropna()
    color = COLORS[sp]

    (osm, osr), (slope, intercept, _) = stats.probplot(sub, dist="norm")
    ax.scatter(osm, osr, color=color, alpha=0.75, s=28,
               edgecolors="white", linewidths=0.4)
    x_line = np.array([osm.min(), osm.max()])
    ax.plot(x_line, slope * x_line + intercept,
            color=C_DARK, linewidth=1.8, linestyle="--", label="Theoretical line")

    W, p = shapiro(sub)
    ax.set_title(f"{sp} | {sx}\n{var}  (n={len(sub)}, p={p:.4f})",
                 fontsize=10, fontweight="bold")
    ax.set_xlabel("Theoretical quantiles")
    ax.set_ylabel("Sample quantiles")
    ax.legend(fontsize=8, frameon=False)
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("Q-Q Plots — cases with p < 0.05 (Shapiro-Wilk)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

### Cases Rejecting H₀ (p < 0.05)

| Species | Sex | Variable | p-value | Diagnosis |
|---|---|---|---|---|
| Adélie | MALE | bill_depth_mm | 0.034 | Borderline — p just below 0.05, n ~ 73. Mild deviation in the tail. |
| Chinstrap | FEMALE | bill_length_mm | 0.002 | 3 outliers in n=34. Skewness=1.39, Kurtosis=4.57. |
| Gentoo | MALE | bill_length_mm | 0.005 | Heavy right tail. |

### Why These Cases Do Not Invalidate the Subsequent Analysis

1. **Wrong variables:** The t-test (Section 3) and ANOVA (Section 5) use `body_mass_g` and `flipper_length_mm` — both normal in all subgroups. The non-normal variables are `bill_length_mm` and `bill_depth_mm`, which are not the dependent variable of any parametric test.

2. **Central Limit Theorem:** For subgroups with n ≥ 30, the sampling distribution of the mean converges to normal regardless of the original distribution — making the t-test and ANOVA robust to these mild violations.

**Conclusion:** The normality assumption holds where it matters. We can proceed safely.

## 3. How Heavy Are the Penguins, Really?
### 95% Confidence Intervals

We have a sample of 333 penguins, not the complete population. The mean observed in the sample is a **point estimate** of the true population mean — but every estimate carries uncertainty.

The **95% confidence interval** quantifies that uncertainty:

> *"We are 95% confident that the true population mean lies within this range."*

**Formula:**

$$\bar{x} \pm t_{\alpha/2,\, n-1} \cdot \frac{s}{\sqrt{n}}$$

| Symbol | Meaning |
|---|---|
| x̄ | Sample mean |
| t | Critical value of the t-Student with n−1 degrees of freedom |
| s | Sample standard deviation |
| n | Sample size |

**Why t-Student and not z?** Because we do not know the true population standard deviation (σ) — we only have the sample estimate s. The t distribution adjusts for that additional uncertainty, especially with small samples.

**How to read the interval:**
- Narrow CI → precise estimate (large sample or homogeneous data)
- Wide CI → high uncertainty (small sample or very dispersed data)
- Two non-overlapping CIs → the difference is likely statistically significant

In [ ]:
def confidence_interval_95(data):
    """
    Computes the 95% CI using t-Student (unknown population variance).
    Returns: (mean, lower_bound, upper_bound)
    """
    n      = len(data)
    mean   = data.mean()
    se     = stats.sem(data)              # standard error = s / sqrt(n)
    t_crit = stats.t.ppf(0.975, df=n-1)  # t for alpha/2 = 0.025
    margin = t_crit * se
    return mean, mean - margin, mean + margin


# Compute the CI for body_mass_g in each species × sex subgroup
results = []
for sp in SPECIES_ORDER:
    for sx in ["MALE", "FEMALE"]:
        sub = df[(df["species"] == sp) & (df["sex"] == sx)]["body_mass_g"]
        mean, lb, ub = confidence_interval_95(sub)
        results.append({
            "Species" : sp, "Sex": sx, "n": len(sub),
            "Mean (g)" : round(mean, 1),
            "CI lower" : round(lb, 1),
            "CI upper" : round(ub, 1),
            "Width"    : round(ub - lb, 1)
        })

ic_df = pd.DataFrame(results)
print("95% Confidence Intervals — body mass (g)\n")
display(ic_df)


In [ ]:
# Plot: violins + individual points + 95% CI
fig, ax = plt.subplots(figsize=(13, 6))

sex_offset = {"MALE": -0.18, "FEMALE": 0.18}
sex_marker = {"MALE": "D",   "FEMALE": "o"}
sex_alpha  = {"MALE": 0.90,  "FEMALE": 0.55}
sex_label  = {"MALE": "Male", "FEMALE": "Female"}

x_ticks, x_labels = [], []

for i, sp in enumerate(SPECIES_ORDER):
    for sx in ["MALE", "FEMALE"]:
        sub   = df[(df["species"] == sp) & (df["sex"] == sx)]["body_mass_g"]
        color = COLORS[sp]
        xpos  = i + sex_offset[sx]

        # Violin
        parts = ax.violinplot(sub, positions=[xpos], widths=0.32,
                              showmeans=False, showmedians=False, showextrema=False)
        for pc in parts["bodies"]:
            pc.set_facecolor(color)
            pc.set_alpha(sex_alpha[sx] * 0.22)
            pc.set_edgecolor(color)
            pc.set_linewidth(0.6)

        # Individual points with jitter
        jitter = np.random.default_rng(42).uniform(-0.06, 0.06, size=len(sub))
        ax.scatter(xpos + jitter, sub, color=color, alpha=0.28, s=12,
                   marker=sex_marker[sx], edgecolors="none", zorder=2)

        # 95% CI
        n      = len(sub)
        mean   = sub.mean()
        se     = stats.sem(sub)
        t_crit = stats.t.ppf(0.975, df=n - 1)
        lb, ub = mean - t_crit * se, mean + t_crit * se

        ax.plot([xpos, xpos], [lb, ub], color=C_DARK, linewidth=2.2,
                zorder=4, solid_capstyle="round")
        for y in [lb, ub]:
            ax.plot([xpos - 0.06, xpos + 0.06], [y, y],
                    color=C_DARK, linewidth=1.8, zorder=4)
        ax.scatter([xpos], [mean], color="white", s=55, zorder=6,
                   edgecolors=color, linewidths=2.0)

        ax.text(xpos + 0.09, ub + 28,  f"{ub:.0f}",   fontsize=7.5,
                color=C_MID, va="bottom", ha="left")
        ax.text(xpos + 0.09, mean,    f"{mean:.0f}", fontsize=8,
                color=color, va="center", ha="left", fontweight="bold")
        ax.text(xpos + 0.09, lb - 28,  f"{lb:.0f}",   fontsize=7.5,
                color=C_MID, va="top", ha="left")

        x_ticks.append(xpos)
        x_labels.append(f"{sex_label[sx]}\n(n={n})")

# Species labels at the top
for i, sp in enumerate(SPECIES_ORDER):
    ax.text(i, 6420, sp, ha="center", fontsize=11,
            fontweight="bold", color=COLORS[sp])
    if i < 2:
        ax.axvline(i + 0.5, color=C_MID, linewidth=0.8, linestyle="--", alpha=0.5)

ax.set_xticks(x_ticks)
ax.set_xticklabels(x_labels, fontsize=9)
ax.set_ylabel("Body mass (g)")
ax.set_xlim(-0.55, 2.55)
ax.set_ylim(2300, 6600)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.spines[["top", "right"]].set_visible(False)

legend = [
    Line2D([0],[0], marker="o", linestyle="None", color=C_MID,
           markersize=7, markerfacecolor="white", markeredgewidth=2, label="Mean"),
    Line2D([0],[0], color=C_DARK, linewidth=2, label="95% CI"),
    Line2D([0],[0], marker="D", linestyle="None", color=C_MID, markersize=6, label="Males"),
    Line2D([0],[0], marker="o", linestyle="None", color=C_MID,
           markersize=6, alpha=0.5, label="Females"),
]
ax.legend(handles=legend, loc="lower right", fontsize=9)
ax.set_title("Body mass by species and sex — distribution + 95% CI\n"
             "Violin = full spread  ·  Points = individual penguins  "
             "·  Bar = 95% CI of the mean",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

### Conclusion — Confidence Intervals

Male Gentoo penguins are the heaviest, with an estimated mean of ~5,484 g. The 95% CI for Gentoo does not overlap with that of any other species, already anticipating that the differences are statistically significant — formally confirmed with the ANOVA in Section 5.

**Why is the CI useful beyond the p-value?** The p-value only answers *"is the difference real?"*. The CI also answers *"how large is it and how precisely do we know?"*. With very large samples it is possible to obtain p < 0.05 for biologically irrelevant differences; the CI puts that magnitude in perspective.

> **Note:** The widest CIs correspond to Chinstrap females (n=34, the smallest group), directly illustrating how sample size affects estimation precision.

## 4. Does Each Species Have Its Own Island?
### Chi-Square Test — Species × Island Association

The chi-square test is the appropriate test to verify whether two **categorical variables** are independent or associated.

We cannot use the t-test or ANOVA here because both require a continuous numeric variable. Species and island are categories.

**How it works:**
1. Build the contingency table with observed frequencies
2. Calculate expected frequencies if the variables were independent: `f_expected = (row total × column total) / grand total`
3. The χ² statistic measures how far the observed values deviate from expected: `χ² = sum of (observed − expected)² / expected`
4. Large χ² → frequencies diverge greatly → association exists

**Hypotheses:**
- H₀: species and island are independent (random distribution)
- H₁: there is an association between species and island
- α = 0.05 | Degrees of freedom = (3−1) × (3−1) = 4

In [ ]:
# Contingency table — observed frequencies
ct = pd.crosstab(df["species"], df["island"])

print("OBSERVED frequencies:\n")
print(ct)
print(f"\nTotal: {ct.values.sum()} penguins")

In [ ]:
# Pearson's chi-square test
chi2, p_chi, dof, expected = chi2_contingency(ct)

print("=== Chi-square test: species × island ===")
print(f"  chi²            : {chi2:.2f}")
print(f"  Degrees of freedom: {dof}")
print(f"  p-value         : {p_chi:.2e}")
print()
decision = "Reject H₀ — association exists" if p_chi < ALPHA else "Fail to reject H₀"
print(f"  Decision        : {decision}")
print()
print("EXPECTED frequencies under H₀ (if independent):")
print(pd.DataFrame(expected.round(1), index=ct.index, columns=ct.columns))
print()
print(f"All cells have expected frequency >= 5: {(expected >= 5).all().all()}")

In [ ]:
# Visualization: observed frequencies heatmap
fig, ax = plt.subplots(figsize=(6, 4))

sns.heatmap(ct, annot=True, fmt="d", cmap="YlGnBu",
            ax=ax, linewidths=0.5, linecolor="white",
            cbar_kws={"shrink": 0.8})
ax.set_title("Observed frequencies", fontweight="bold")
ax.set_xlabel("Island")
ax.set_ylabel("Species")

plt.tight_layout()
plt.show()


### Conclusion — Chi-Square

**Enormous χ², p ~ 0.** We reject H₀ with near-absolute certainty.

The distribution of species across islands is not random. Each species has a marked geographic preference:

| Species | Dominant island | Notes |
|---|---|---|
| Adélie | Torgersen | Only species present — strongly positive residual |
| Gentoo | Biscoe | Concentrated almost exclusively here |
| Chinstrap | Dream | Dominates this island |

Standardized residuals with values above |5| confirm that these differences are not marginal — they are structural.

This geographic separation is the starting point for the question in the next section: if the species live apart, does that mean the geographic environment shapes their body?

## 5. Does the Island Affect the Weight of Adélie Penguins?
### One-Way ANOVA — Body Mass by Island

We just confirmed that each species lives on its own island. Now we flip the question: **does it matter where you live, if you are the same species?**

Adélie penguins are the only species present on all three islands — Biscoe, Dream, and Torgersen. This gives us a **natural experiment**: same species, three different environments. If weight varies by island, it is evidence that geographic isolation has a real biological effect.

**Why ANOVA and not three t-tests?**

With three groups we would need three separate comparisons. Each t-test has a 5% probability of a false positive. Accumulated, the total error rises to ~14%. The ANOVA compares all three groups in a single test, keeping the global α = 0.05. If the ANOVA is significant, we use Tukey HSD to see which specific pair differs.

**Hypotheses:**
- H₀: μ_Biscoe = μ_Dream = μ_Torgersen — island does not affect mass
- H₁: at least one island has a different mean
- α = 0.05

**Prerequisites verified:**
- Normality: verified in Section 2 for Adélie
- Homogeneity of variances: verified now with Levene's test

In [ ]:
# Filter only Adélie — the only species on all 3 islands
adelie = df[df["species"] == "Adelie"]

print("Adélie by island — descriptive statistics:")
print(adelie.groupby("island")["body_mass_g"]
      .agg(["count", "mean", "std", "min", "max"])
      .round(1))
print()

# STEP 1: Levene's test — homogeneous variances?
# H₀: the variances of the three groups are equal
island_groups = [adelie[adelie["island"] == isl]["body_mass_g"]
                 for isl in sorted(adelie["island"].unique())]

lev_stat, lev_p = levene(*island_groups)
print("=== Levene's Test (homogeneity of variances) ===")
print(f"  Statistic: {lev_stat:.4f}  |  p-value: {lev_p:.4f}")
print(f"  Result   : {'Variances NOT homogeneous' if lev_p < ALPHA else 'Homogeneous variances — ANOVA appropriate'}")
print()

# STEP 2: One-way ANOVA
f_stat, p_anova = f_oneway(*island_groups)
print("=== ANOVA — body_mass_g ~ island (Adélie only) ===")
print(f"  F-statistic: {f_stat:.4f}")
print(f"  p-value    : {p_anova:.4f}")
print(f"  Decision   : {'Reject H₀ — some island differs' if p_anova < ALPHA else 'Fail to reject H₀ — no island differs significantly'}")
print()

# STEP 3: Tukey HSD — pairwise comparisons
tukey = pairwise_tukeyhsd(endog=adelie["body_mass_g"],
                          groups=adelie["island"], alpha=ALPHA)
print("=== Tukey HSD — pairwise comparisons ===")
print(tukey)

In [ ]:
# Visualization: full distribution + means with CI by island
islands = sorted(adelie["island"].unique())

island_colors = {
    "Biscoe"   : "#5B8DB8",
    "Dream"    : "#6BAE8C",
    "Torgersen": "#B07DB0",
}

fig, ax = plt.subplots(figsize=(9, 5))

for i, isl in enumerate(islands):
    sub   = adelie[adelie["island"] == isl]["body_mass_g"]
    color = island_colors[isl]

    parts = ax.violinplot(sub, positions=[i], widths=0.45,
                          showmeans=False, showmedians=False, showextrema=False)
    for pc in parts["bodies"]:
        pc.set_facecolor(color); pc.set_alpha(0.22); pc.set_edgecolor(color)

    jitter = np.random.default_rng(42).uniform(-0.07, 0.07, size=len(sub))
    ax.scatter(i + jitter, sub, color=color, alpha=0.42, s=16,
               edgecolors="none", zorder=3)

    n      = len(sub)
    mean   = sub.mean()
    t_crit = stats.t.ppf(0.975, df=n - 1)
    lb, ub = mean - t_crit * stats.sem(sub), mean + t_crit * stats.sem(sub)

    ax.plot([i, i], [lb, ub], color=C_DARK, linewidth=2.5,
            zorder=5, solid_capstyle="round")
    for y in [lb, ub]:
        ax.plot([i - 0.07, i + 0.07], [y, y], color=C_DARK, linewidth=1.8, zorder=5)
    ax.scatter([i], [mean], color="white", s=55, zorder=6,
               edgecolors=color, linewidths=2.0)

    ax.text(i + 0.10, ub + 25,  f"{ub:.0f}",   fontsize=8, color=C_MID, va="bottom")
    ax.text(i + 0.10, mean,    f"{mean:.0f}", fontsize=8.5, color=color,
            va="center", fontweight="bold")
    ax.text(i + 0.10, lb - 25,  f"{lb:.0f}",   fontsize=8, color=C_MID, va="top")

ax.set_xticks(range(len(islands)))
ax.set_xticklabels([f"{isl}\n(n={len(adelie[adelie['island']==isl])})"
                    for isl in islands])
ax.set_ylabel("Body mass (g)")
ax.set_title("Adélie body mass by island\nDistribution + 95% CI  ·  circle = mean",
             fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

### Conclusion — ANOVA Adélie by Island

**The ANOVA does not reject H₀ (p > 0.05).** There are no significant differences in body mass of Adélie penguins across the three islands.

The observed means — ~3,710 g on Biscoe, ~3,701 g on Dream, ~3,709 g on Torgersen — differ by less than 10 g from each other, statistically indistinguishable from sampling noise. Tukey HSD confirms that no pair of islands is significantly different.

**What does this "non-difference" tell us?**

Adélie penguins are a physically robust species with respect to geographic environment. Living on Biscoe, Dream, or Torgersen leaves no measurable imprint on weight.

This contrasts with what we saw in Section 4: the island determines *who* lives where, but not *how* they grow. The habitat selects the species, but does not shape the body within it.

> **Note:** If island were included as a predictor alongside all species, it would appear significant — not because it causes differences in weight, but because it proxies species membership (Gentoo, the heaviest, live almost exclusively on Biscoe). This is the same confounding mechanism seen in Section 6.

## 6. Does the Bill Reveal the Species? Simpson's Paradox
### Pearson Correlation — Simpson's Paradox

We compute the Pearson correlation between bill length and bill depth. A straightforward question: do penguins with longer bills also have deeper bills?

**The answer depends entirely on how we look at the data.**

Pearson's correlation (r) measures the strength and direction of a linear relationship:

| r value | Interpretation |
|---|---|
| r ~ +1 | Perfect positive relationship |
| r ~ 0 | No linear relationship |
| r ~ −1 | Perfect negative relationship |

> **Important:** r measures association, not causality.

**The experiment:** We compute r in two ways:
1. Across all penguins mixed together — without distinguishing species
2. Within each species separately

If statistics were neutral to the level of aggregation, the results should be consistent. They are not — and that inconsistency is the lesson.

**Why does the paradox occur here?**
Gentoo penguins have a long but shallow bill. Adélie penguins have a short but deep bill. When you mix the three groups, the between-species pattern dominates the within-species pattern, **reversing the apparent correlation**.

The species is a **confounding variable** — a hidden variable that, if not controlled, completely distorts the conclusion.

In [ ]:

# Global correlation — all penguins together
r_total, p_total = stats.pearsonr(df["bill_length_mm"], df["bill_depth_mm"])
print("GLOBAL correlation (all penguins together):")
display(pd.DataFrame([{
    "Group"    : "TOTAL",
    "r"        : round(r_total, 4),
    "p-value"  : f"{p_total:.4e}",
    "Direction": "Negative" if r_total < 0 else "Positive"
}]))

# Correlation by species
print("\nCorrelation by SPECIES (disaggregated data):")
rows = []
for sp in sorted(df["species"].unique()):
    sub = df[df["species"] == sp]
    r_sp, p_sp = stats.pearsonr(sub["bill_length_mm"], sub["bill_depth_mm"])
    rows.append({
        "Species"  : sp,
        "r"        : round(r_sp, 4),
        "p-value"  : f"{p_sp:.4e}",
        "Direction": "Positive" if r_sp > 0 else "Negative"
    })
display(pd.DataFrame(rows))

print("\nPARADOX: globally NEGATIVE, within each species POSITIVE.")

In [ ]:
# Simpson's Paradox visualization — two perspectives
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Global sample — apparent negative correlation
ax = axes[0]
ax.scatter(df["bill_length_mm"], df["bill_depth_mm"],
           c=[COLORS[sp] for sp in df["species"]], alpha=0.55, s=28)
m, b = np.polyfit(df["bill_length_mm"], df["bill_depth_mm"], 1)
x_line = np.linspace(df["bill_length_mm"].min(), df["bill_length_mm"].max(), 100)
ax.plot(x_line, m * x_line + b, color=C_DARK, linewidth=2, linestyle="--",
        label=f"Global line (r = {r_total:.2f})")
ax.set_xlabel("Bill length (mm)")
ax.set_ylabel("Bill depth (mm)")
ax.set_title("Global sample — NEGATIVE correlation (paradox)", fontweight="bold")
ax.legend(frameon=False, fontsize=9)
ax.spines[["top", "right"]].set_visible(False)

# 2. By species — each group shows positive correlation
ax = axes[1]
for sp in sorted(df["species"].unique()):
    sub = df[df["species"] == sp]
    ax.scatter(sub["bill_length_mm"], sub["bill_depth_mm"],
               color=COLORS[sp], alpha=0.55, s=28, label=sp)
    m_sp, b_sp = np.polyfit(sub["bill_length_mm"], sub["bill_depth_mm"], 1)
    x_sp = np.linspace(sub["bill_length_mm"].min(), sub["bill_length_mm"].max(), 50)
    ax.plot(x_sp, m_sp * x_sp + b_sp, color=COLORS[sp], linewidth=2)
ax.set_xlabel("Bill length (mm)")
ax.set_ylabel("Bill depth (mm)")
ax.set_title("By species — POSITIVE correlation within each group", fontweight="bold")
ax.legend(frameon=False, fontsize=9)
ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("Simpson's Paradox — Penguin bill dimensions\n"
             "Species is a confounding variable that reverses the apparent correlation",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


### Conclusion — Simpson's Paradox

The correlation between bill length and bill depth is **negative globally** (r ~ −0.23), but **positive within each species** (r between +0.35 and +0.65).

The species acts as a **confounding variable**: its between-group structure reverses the apparent correlation when all data are mixed.

**Practical lesson:**

> Never interpret a correlation on aggregated data without first verifying whether there are subgroups with distinct dynamics.

If someone took the global data and concluded that "penguins with longer bills have shallower bills", they would be wrong. The correct conclusion is the opposite — but it only becomes visible when disaggregating by species.

This lesson applies well beyond penguins: it is one of the most common statistical biases in medical, economic, and social studies.

## 7. Can We Tell Which Species a Penguin Belongs To Just by Looking at It?
### Logistic Regression — Identifying Gentoo vs. Non-Gentoo

Gentoo penguins live almost exclusively on Biscoe. If their physical characteristics are so distinct — as seen in the EDA — a model should be able to recognize them with high reliability just by looking at their body.

We focus on the binary classification **Gentoo vs. Non-Gentoo**.

**Why logistic regression and not linear regression?**

Linear regression predicts a continuous number (weight in grams). Here we want to predict a **probability** — "how likely is this penguin to be a Gentoo?" — a number between 0 and 1.

The sigmoid function transforms any real value into a probability:

$$P(\text{Gentoo}) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 x_1 + \beta_2 x_2)}}$$

If P > 0.5 → classify as Gentoo. If P ≤ 0.5 → Non-Gentoo.

**Why only 2 variables?**

With all 4 morphological variables, the Gentoo penguins are so well separated from the rest that the model converges to perfect predictions of 0 or 1 — this is called **quasi-perfect separation**. In that case, coefficients become numerically unstable. We use the 2 most informative variables (`flipper_length_mm` and `bill_depth_mm`) to keep the model stable.

**Odds Ratio (OR)**

Logistic regression coefficients are not interpreted directly — they are exponentiated to obtain the OR:

| OR | Interpretation |
|---|---|
| OR > 1 | Increases probability of being Gentoo |
| OR = 1 | No effect |
| OR < 1 | Decreases probability of being Gentoo |

In [ ]:
# Binary target variable: 1 = Gentoo, 0 = Non-Gentoo
df["is_gentoo"] = (df["species"] == "Gentoo").astype(int)
y = df["is_gentoo"].values

FEATS_LOGIT = ["flipper_length_mm", "bill_depth_mm"]

# Train/test split: 75% for training, 25% for evaluation.
# stratify=y maintains the same Gentoo/Non-Gentoo ratio in both sets.
X_logit  = df[FEATS_LOGIT].values
X_train, X_test, y_train, y_test = train_test_split(
    X_logit, y, test_size=0.25, random_state=42, stratify=y)

# StandardScaler: brings variables to the same scale (mean 0, std 1).
# Fit is done ONLY on the training set — the test set must never
# influence transformation parameters (data leakage).
scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Train: {X_train.shape[0]} penguins  |  Test: {X_test.shape[0]} penguins")
print(f"Gentoo in test: {y_test.sum()} of {len(y_test)}")

In [ ]:
# Model with statsmodels to obtain Odds Ratios with confidence intervals.
# statsmodels provides detailed output with coefficients, standard errors, and p-values.
X_sm        = sm.add_constant(X_train_s)
logit_model = sm.Logit(y_train, X_sm).fit(method="bfgs", disp=False)
print(logit_model.summary())

In [ ]:
# Odds Ratio = exp(coefficient)
# Variables are standardized, so OR refers to
# an increase of 1 standard deviation in that variable.
params  = logit_model.params[1:]
ci      = logit_model.conf_int()
ci_low  = ci[1:, 0]
ci_high = ci[1:, 1]

or_df = pd.DataFrame({
    "Variable"  : FEATS_LOGIT,
    "Odds Ratio": np.exp(params).round(3),
    "95% CI low": np.exp(ci_low).round(3),
    "95% CI up" : np.exp(ci_high).round(3),
    "p-value"   : logit_model.pvalues[1:].round(4)
}).reset_index(drop=True)

print("=== Odds Ratios (standardized variables) ===\n")
print(or_df.to_string(index=False))
print()
print("OR >> 1 — increasing flipper_length greatly raises the probability of being Gentoo")
print("OR << 1 — increasing bill_depth lowers the probability of being Gentoo")

In [ ]:
# Classification metrics with sklearn
lr = LogisticRegression(random_state=42)
lr.fit(X_train_s, y_train)
y_pred = lr.predict(X_test_s)

print("=== Classification Report (test set — never-seen data) ===")
print()
print("  precision : of all predicted Gentoo, how many were correct?")
print("  recall    : of all actual Gentoo, how many did the model find?")
print("  F1-score  : harmonic mean of precision and recall")
print()
print(classification_report(y_test, y_pred, target_names=["Non-Gentoo", "Gentoo"]))

In [ ]:
# Confusion matrix — diagonal = correct predictions, off-diagonal = errors
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred),
    display_labels=["Non-Gentoo", "Gentoo"]
).plot(ax=ax, colorbar=False,
       cmap=mcolors.LinearSegmentedColormap.from_list(
           "palette", ["white", COLORS["Gentoo"]]))
ax.set_title("Confusion Matrix\ndiagonal = correct predictions",
             fontweight="bold")
plt.tight_layout()
plt.show()


### Conclusion — Logistic Regression

The model achieves **perfect classification** on the test set (accuracy = 100%).

With only two morphological variables — `flipper_length_mm` and `bill_depth_mm` — we can identify whether a penguin is Gentoo or not with zero errors on unseen data.

This confirms what the EDA already suggested: **Gentoo penguins occupy a completely distinct morphological space** from Adélie and Chinstrap. Longer flippers and shallower bills are reliable biological signatures of the species.

> **Caveat:** Perfect classification on a dataset this clean may not generalize to messier real-world data. The quasi-perfect separation also makes coefficient interpretation unreliable — the model works, but the OR values should not be over-interpreted.

---

## Summary

| Section | Test | Key Result |
|---|---|---|
| 1 | EDA | Gentoo are heavier and have longer flippers; Adélie have shorter bills |
| 2 | Shapiro-Wilk | Normality holds for all dependent variables used in tests |
| 3 | 95% CI | Non-overlapping CIs confirm species-level mass differences |
| 4 | Chi-square | χ² = 284.59, p ~ 0 — species and island are strongly associated |
| 5 | ANOVA | F = 0.005, p = 0.995 — island has no effect on Adélie body mass |
| 6 | Pearson r | Global r = −0.23 vs. within-species r = +0.35 to +0.65 (Simpson's Paradox) |
| 7 | Logistic Reg. | Perfect classification — Gentoo identified from 2 morphological features |

---
*Dataset: [Palmer Penguins](https://allisonhorst.github.io/palmerpenguins/) · Horst, Hill & Gorman (2020)*